##### Quick notebook to test out Prompt Evaluation.
### *See script in action evaluations/run_eval.py.* 
- Create a prompt
- Create a test dataset
- Create eval scripts to grade answers
- Iterate on prompt
- Repeat process

In [1]:
import json
import yaml
import logging
import os
import sys
from typing import Dict, List
from pathlib import Path

from anthropic import Anthropic
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:


sys.path.insert(0, str(Path(".").resolve()))



In [4]:

def load_prompt(prompt_path: str) -> Dict:
    with open(prompt_path, 'r') as f:
        return yaml.safe_load(f)

def load_eval_set(eval_path: str) -> Dict:
    with open(eval_path, 'r') as f:
        return json.load(f)

In [5]:
from difflib import SequenceMatcher
from typing import Dict
import re

def calculate_translation_similarity(expected: str, actual: str) -> float:
    """
    Compare two translations using multiple strategies.
    Returns a score between 0 and 1.
    """
    # Strategy 1: Exact match (best case)
    if expected.lower().strip() == actual.lower().strip():
        return 1.0
    
    # Strategy 2: Sequence matching (handles minor variations)
    sequence_ratio = SequenceMatcher(None, expected.lower(), actual.lower()).ratio()
    
    # Strategy 3: Word-level overlap (handles reordering)
    expected_words = set(re.findall(r'\b\w+\b', expected.lower()))
    actual_words = set(re.findall(r'\b\w+\b', actual.lower()))
    
    if not expected_words:  # Empty string edge case
        return 1.0 if not actual_words else 0.0
    
    word_overlap = len(expected_words & actual_words) / len(expected_words | actual_words)
    
    # Combine strategies: sequence matching is more strict, word overlap is more lenient
    combined_score = (sequence_ratio * 0.7) + (word_overlap * 0.3)
    
    return combined_score

def calculate_score(expected: Dict, actual: Dict, category: str) -> float:
    """
    Score logic:
    - Italian translation similarity: 0.3 weight
    - French translation similarity: 0.3 weight
    - Sentiment classification: exact match, 0.2 weight
    - Sentence type classification: exact match, 0.15 weight
    - Notes quality: presence check, 0.05 weight
    """
    
    scores = {}
    
    # 1. Italian translation comparison
    if 'italian' in expected and 'italian' in actual:
        italian_score = calculate_translation_similarity(
            expected['italian'],
            actual['italian']
        )
        scores['italian'] = italian_score
    else:
        scores['italian'] = 0.0
    
    # 2. French translation comparison
    if 'french' in expected and 'french' in actual:
        french_score = calculate_translation_similarity(
            expected['french'],
            actual['french']
        )
        scores['french'] = french_score
    else:
        scores['french'] = 0.0
    
    # 3. Sentiment classification (exact match)
    sentiment_match = 1.0 if (
        expected.get('sentiment', '').lower().strip() == 
        actual.get('sentiment', '').lower().strip()
    ) else 0.0
    scores['sentiment'] = sentiment_match
    
    # 4. Sentence type classification (exact match)
    sentence_type_match = 1.0 if (
        expected.get('sentence_type', '').lower().strip() == 
        actual.get('sentence_type', '').lower().strip()
    ) else 0.0
    scores['sentence_type'] = sentence_type_match
    
    # 5. Notes presence (0 or 1 - just checking if notes exist)
    notes_present = 1.0 if actual.get('notes', '').strip() else 0.0
    scores['notes'] = notes_present
    
    # Weighted average
    weights = {
        'italian': 0.30,
        'french': 0.30,
        'sentiment': 0.20,
        'sentence_type': 0.15,
        'notes': 0.05
    }
    
    total_score = sum(scores[key] * weights[key] for key in scores)
    
    return total_score, scores  # Return both total and breakdown

In [6]:
def parse_output(response_text: str) -> Dict:
    """
    Parse LLM response into structured output.
    Expected format: language: word with article
    Example: "French: le mot"
    """
    result = {
        "output_context": None,
        "fr": None,
        "it": None
    }
    
    lines = response_text.strip().split('\n')
    
    for line in lines:
        if 'French' in line or 'fr' in line.lower():
            # Extract French translation
            parts = line.split(':')
            if len(parts) > 1:
                result['fr'] = parts[1].strip()
        elif 'Italian' in line or 'it' in line.lower():
            # Extract Italian translation
            parts = line.split(':')
            if len(parts) > 1:
                result['it'] = parts[1].strip()
    
    # Set output_context (adjust based on your actual prompt structure)
    result['output_context'] = response_text.strip()
    
    return result

In [7]:

def run_evaluation(prompt_config: Dict, eval_set: Dict) -> List[Dict]:

    client = Anthropic(api_key=os.getenv("CLAUDE_COURSE_API_KEY"))
    model_name = "claude-sonnet-4-0"
    results = []
    
    for test_case in eval_set['translations']:
        # Format the user message
        english_text = test_case['english']  
        user_msg = prompt_config['user_template'].format(
            english_text=test_case['english']
        )
        
        # Call the LLM
        response = client.messages.create(
            model=model_name,
            max_tokens=100,
            system=prompt_config['system']['role'],
            messages=[{"role": "user", "content": user_msg}]
        )
        # Score the response
        actual_output = parse_output(response.content[0].text)

        print(f"user_msg: {user_msg}")
        print(f"Actual: {actual_output}")
        # print(f"Score: {score}")

        # Prepare expected output (match your data structure)
        expected_output = {
            'italian': test_case['italian'],
            'french': test_case['french'],
            'sentiment': test_case['sentiment'],
            'sentence_type': test_case['type'],
            'notes': test_case.get('notes', '')
        }
        
        # Score with breakdown
        total_score, score_breakdown = calculate_score(
            expected=expected_output,
            actual=actual_output,
            category=test_case.get('category', 'general')
        )
        
        results.append({
            "english": english_text,
            "expected": expected_output,
            "actual": actual_output,
            "total_score": total_score,
            "score_breakdown": score_breakdown,
            "status": "success",
            "passed": total_score >= 0.75  # Adjust threshold as needed
        })
    
    return results


In [10]:
if __name__ == "__main__":

    prompt = load_prompt("prompts/fr_it_prompt_v1.yml")
    eval_set = load_eval_set("evaluations/fr_it_prompt_eval_v1.json")

    prompt = load_prompt(f"{Path.cwd()}/prompts/fr_it_prompt_v1.yml")
    eval_set = load_eval_set(f"{Path.cwd()}/evaluations/fr_it_prompt_eval_v1.json") 
    
    results = run_evaluation(prompt, eval_set)

        # Save results
    Path("evaluations/results").mkdir(parents=True, exist_ok=True)
    with open("evaluations/results/latest_eval.json", "w") as f:
        json.dump(results, f, indent=2)

    # Print summary
    passed = sum(1 for r in results if r.get('passed', False))
    print(f"✓ Evaluation complete: {passed}/{len(results)} passed")

INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


user_msg: Translate the following english text:
The book is on the table.

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Il libro è sul tavolo.\n\n**French:** Le livre est sur la table.', 'fr': '** Le livre est sur la table.', 'it': '** Il libro è sul tavolo.'}


INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


user_msg: Translate the following english text:
She is my sister.

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Lei è mia sorella.\n\n**French:** Elle est ma sœur.', 'fr': '** Elle est ma sœur.', 'it': '** Lei è mia sorella.'}


INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...

user_msg: Translate the following english text:
Cringe

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Cringe (often used as is, or "imbarazzante" / "che fa venire i brividi")\n\n**French:** Cringe (often used as is, or "gênant" / "embarrassant")\n\nNote: "Cringe" is commonly used as-is in both Italian and French, especially among younger speakers and in internet contexts. The alternative translations provided capture the meaning of something being embarrass', 'fr': '"Cringe" is commonly used as-is in both Italian and French, especially among younger speakers and in internet contexts. The alternative translations provided capture the meaning of something being embarrass', 'it': '** Cringe (often used as is, or "imbarazzante" / "che fa venire i brividi")'}


INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


user_msg: Translate the following english text:
Cozy

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Accogliente\n\n**French:** Douillet / Confortable\n\nNote: In French, "douillet" emphasizes the warm, snug feeling, while "confortable" is more general for comfortable. Both can be used depending on the specific context.', 'fr': 'In French, "douillet" emphasizes the warm, snug feeling, while "confortable" is more general for comfortable. Both can be used depending on the specific context.', 'it': '** Accogliente'}


INFO: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


user_msg: Translate the following english text:
Oversharing

Actual: {'output_context': 'Here are the translations:\n\n**Italian:** Condivisione eccessiva\n\n**French:** Partage excessif\n\nBoth translations capture the concept of sharing too much personal information, typically in social contexts or on social media.', 'fr': '** Partage excessif', 'it': '** Condivisione eccessiva'}
✓ Evaluation complete: 0/11 passed
